# 02 - Silver | limpeza, padronizacao e classificacao de dado de saude

O Pitaia armazena prontuario, exame, medicacao e metrica corporal. Isso e dado
pessoal **sensivel** pela LGPD (art. 5, II), com regime proprio no art. 11.
Entao a classificacao aqui nao pode ser genericamente "PII": ela precisa separar
dado pessoal comum de dado de saude, porque o risco e a base legal dos dois sao
diferentes.

Perfilo a qualidade **antes** de limpar. Medir depois nao prova nada.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "workspace"
BRONZE   = f"{CATALOGO}.bronze"
SILVER   = f"{CATALOGO}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER} COMMENT "
          "'Catalogo de seguranca do Pitaia limpo, tipado e com classificacao LGPD. "
          "Identificadores anonimizados para publicacao.'")

def bronze(t):
    return spark.table(f"{BRONZE}.{t}")

def sql_txt(t):
    """Aspa simples no texto quebra o literal SQL. Dobrar e o escape."""
    return t.replace("'", "''")

def salvar(df, nome, doc):
    alvo = f"{SILVER}.{nome}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(alvo)
    spark.sql(f"COMMENT ON TABLE {alvo} IS '{sql_txt(doc)}'")
    print(f"{alvo:46s} {spark.table(alvo).count():6d}")
    return spark.table(alvo)

# o export do Supabase devolve booleano como t/f, mas pela UI as vezes vem true/false
def booleano(coluna):
    return F.lower(F.trim(F.col(coluna))).isin("t", "true", "1")

## Perfil de qualidade na Bronze

In [0]:
TABELAS = ["objetos", "politicas", "colunas", "grants", "funcoes",
           "buckets", "extensoes", "constraints"]

perfil = []
for t in TABELAS:
    df = bronze(t)
    total = df.count()
    for c in [x for x in df.columns if not x.startswith("_")]:
        vazio = F.col(c).isNull() | (F.trim(F.col(c)) == "")
        r = df.agg(F.sum(F.when(vazio, 1).otherwise(0)).alias("n"),
                   F.countDistinct(c).alias("d"),
                   F.max(F.length(c)).alias("tam")).collect()[0]
        perfil.append((t, c, total, int(r["n"]),
                       round(100 * r["n"] / total, 2) if total else None,
                       int(r["d"]), r["tam"]))

df_perfil = spark.createDataFrame(perfil, """tabela string, coluna string, n_linhas int,
    n_nulos int, pct_nulo double, n_distintos int, tamanho_max int""")

salvar(df_perfil, "perfil_qualidade",
       "Completude, cardinalidade e tamanho por atributo, medidos na Bronze antes de "
       "qualquer limpeza. Evidencia da etapa 4.5 do MVP.")
display(df_perfil.orderBy(F.desc("pct_nulo")))

workspace.silver.perfil_qualidade                  44


tabela,coluna,n_linhas,n_nulos,pct_nulo,n_distintos,tamanho_max
buckets,file_size_limit,7,7,100.0,0,null
objetos,opcoes,344,341,99.13,1,21
objetos,comentario,344,327,95.06,17,106
colunas,column_default,3311,2877,86.89,77,56
funcoes,config,195,151,77.44,5,30
politicas,expressao_check,209,140,66.99,36,794
politicas,expressao_using,209,36,17.22,84,794
objetos,schema_nome,344,0,0.0,12,19
objetos,objeto,344,0,0.0,342,37
objetos,tipo_bruto,344,0,0.0,3,1


## Duplicatas por chave natural

In [0]:
CHAVES = {
    "objetos":     ["schema_nome", "objeto"],
    "politicas":   ["schema_nome", "objeto", "politica"],
    "colunas":     ["schema_nome", "objeto", "coluna"],
    "grants":      ["role_nome", "schema_nome", "objeto", "privilegio"],
    "funcoes":     ["schema_nome", "funcao"],
    "constraints": ["schema_nome", "objeto", "constraint_nome"],
}

dups = []
for t, chave in CHAVES.items():
    n = bronze(t).count()
    u = bronze(t).select(*chave).distinct().count()
    dups.append((t, ",".join(chave), n, u, n - u))

salvar(spark.createDataFrame(dups, """tabela string, chave_natural string,
       n_linhas int, n_unicas int, n_duplicatas int"""),
       "perfil_duplicatas",
       "Duplicatas por chave natural medidas antes da deduplicacao. Dimensao de unicidade.")

display(spark.table(f"{SILVER}.perfil_duplicatas"))

workspace.silver.perfil_duplicatas                  6


tabela,chave_natural,n_linhas,n_unicas,n_duplicatas
objetos,"schema_nome,objeto",344,344,0
politicas,"schema_nome,objeto,politica",209,209,0
colunas,"schema_nome,objeto,coluna",3311,3311,0
grants,"role_nome,schema_nome,objeto,privilegio",5707,5707,0
funcoes,"schema_nome,funcao",195,165,30
constraints,"schema_nome,objeto,constraint_nome",380,380,0


## Objetos

Nao removo os schemas internos do Supabase. Marco eles com `dominio` e uso como
grupo de comparacao na analise - sem uma referencia, dizer "tenho 14 achados" nao
significa nada.

A anonimizacao existe porque o repositorio e publico e o Pitaia esta em producao
com prontuario de gente real. Publicar o nome da tabela vulneravel seria entregar
o caminho pronto.

In [0]:
SUPABASE = ["auth", "storage", "realtime", "vault", "extensions", "graphql",
            "graphql_public", "supabase_functions", "supabase_migrations",
            "pgbouncer", "cron", "net", "_realtime", "pgsodium", "pgsodium_masks"]
SISTEMA  = ["pg_catalog", "information_schema", "pg_toast"]

# conferir em Supabase > Settings > API > Exposed schemas. O padrao e so public.
EXPOSTOS = ["public", "graphql_public"]

tipo = (F.when(F.col("tipo_bruto") == "r", "tabela")
         .when(F.col("tipo_bruto") == "p", "tabela_particionada")
         .when(F.col("tipo_bruto") == "v", "view")
         .when(F.col("tipo_bruto") == "m", "view_materializada")
         .when(F.col("tipo_bruto") == "f", "tabela_estrangeira")
         .otherwise("outro"))

obj = (bronze("objetos")
       .filter(~F.col("schema_nome").isin(SISTEMA))
       .withColumn("tipo_objeto", tipo)
       .withColumn("rls_habilitado", booleano("rls_habilitado"))
       .withColumn("rls_forcado", booleano("rls_forcado"))
       .withColumn("dominio", F.when(F.col("schema_nome").isin(SUPABASE),
                                     "gerenciado_supabase").otherwise("aplicacao"))
       .withColumn("exposto_api", F.col("schema_nome").isin(EXPOSTOS))
       .withColumn("security_invoker",
                   F.coalesce(F.col("opcoes"), F.lit("")).contains("security_invoker=true"))
       .withColumn("tem_documentacao", F.col("comentario").isNotNull())
       .dropDuplicates(["schema_nome", "objeto"]))

ordem = Window.orderBy("schema_nome", "objeto")
obj = (obj
       .withColumn("objeto_anon",
                   F.concat(F.lit("t_"), F.lpad(F.row_number().over(ordem).cast("string"), 3, "0")))
       .withColumn("schema_anon",
                   F.when(F.col("dominio") == "aplicacao",
                          F.concat(F.lit("app_s"),
                                   F.dense_rank().over(Window.orderBy("schema_nome")).cast("string")))
                    .otherwise(F.col("schema_nome"))))

salvar(obj.select("schema_nome", "schema_anon", "objeto", "objeto_anon", "tipo_objeto",
                  "dominio", "exposto_api", "rls_habilitado", "rls_forcado",
                  "security_invoker", "tem_documentacao", "_snapshot"),
       "objeto",
       "Um registro por objeto do banco. dominio separa o que e da aplicacao do que o "
       "Supabase gerencia. exposto_api indica alcance via PostgREST. Origem: bronze.objetos.")

salvar(obj.select("schema_nome", "objeto", "schema_anon", "objeto_anon"),
       "_de_para_nao_publicar",
       "PRIVADO. De-para entre nome real e anonimizado. Nao versionar.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.silver.objeto                           133
workspace.silver._de_para_nao_publicar            133


DataFrame[schema_nome: string, objeto: string, schema_anon: string, objeto_anon: string]

## Colunas e a classificacao LGPD

Tres niveis, nao dois:

- `saude` e `biometrico` -> dado sensivel, art. 5 II + art. 11
- `credencial`, `documento`, `financeiro` -> dado pessoal de alto impacto
- `email`, `telefone`, `endereco`, `nome_pessoa` -> dado pessoal comum

Decisao interpretativa que preciso saber defender: **peso, altura, IMC,
frequencia cardiaca e sono entram como dado de saude.** Isoladamente pareceriam
antropometria banal, mas coletados por uma plataforma de saude, com medico e
educador fisico acessando, sao "dado referente a saude" no sentido do art. 5 II.
Classifiquei pelo contexto de tratamento, nao pelo nome do campo.

In [0]:
# O schema do Pitaia e em ingles, entao os padroes sao em ingles com alguns termos
# em portugues por seguranca.
#
# Classifico por TABELA e por COLUNA. Isso importa: uma coluna chamada "notes" dentro
# de medical_record_notes e dado clinico; a mesma coluna "notes" em checkin_groups nao
# e. Olhar so o nome da coluna perde a maior parte do dado de saude - na primeira
# versao eu achava 12 colunas em 640, o que era obviamente errado para uma plataforma
# de saude. Com heranca de tabela sao 301.

TABELA_CLASSE = [
    ("saude_clinica", r"medical_record|anamnesis|lab_(result|marker)|medication|menstrual|"
                      r"anxiety|crisis|diagnos|symptom|treatment|prescription|patient_file|"
                      r"diary_entr|self_assessment|questionnaire_answer|prontuario|exame"),
    ("saude_metrica", r"body_measurement|wearable_metric|exercise_log|workout_log|"
                      r"workout_feedback|\bmeals?\b|nutrition|photo_tracking|checkin|"
                      r"\bgoals?\b|alert_completion|assigned_(workout|questionnaire)"),
    ("credencial",    r"ai_keys|otp|trusted_device|unsubscribe_token|invite_link"),
]

COLUNA_CLASSE = [
    ("credencial",    r"senha|password|secret|token|api_?key|otp|refresh"),
    ("saude_clinica", r"diagnos|symptom|medication|dosage|prescri|allerg|anamnes|lab_|marker|"
                      r"clinical|medical|crisis|severity|mood|pain|sleep_quality|cid10"),
    ("saude_metrica", r"weight|height|\bbmi\b|body_fat|muscle|circumference|heart_rate|\bbpm\b|"
                      r"spo2|steps|calories|hydration|glucose|sleep|intensity|reps|duration_min"),
    ("biometrico",    r"biometr|face_|fingerprint"),
    ("documento",     r"\bcpf\b|\bcnpj\b|\brg\b|passport|\bcnh\b|\bcns\b|document_number"),
    ("prof_saude",    r"\bcrm\b|crefito|\bcref\b|council|license_number|specialty"),
    ("financeiro",    r"card_|iban|\bpix\b|bank_|invoice"),
    ("email",         r"e_?mail"),
    ("telefone",      r"phone|whatsapp|mobile_number|telefone|celular"),
    ("endereco",      r"address|zip_?code|postal|street|neighborhood|\bcep\b"),
    ("nascimento",    r"birth|\bdob\b|nascimento|\bage\b"),
    ("nome_pessoa",   r"^name$|full_name|first_name|last_name|display_name|patient_name"),
    ("localizacao",   r"latitude|longitude|\blat\b|\blng\b|geoloc"),
]

# colunas de controle nao herdam a sensibilidade da tabela: um created_at de
# medical_record_notes nao e dado de saude
TECNICAS = r"^(id|created_at|updated_at|deleted_at|.*_id|sort_order|is_active|version|status|type|created_by|updated_by)$"

SAUDE = ["saude_clinica", "saude_metrica"]
SENSIVEIS = SAUDE + ["biometrico", "origem_racial", "religiao"]

col_lower = F.lower(F.col("coluna"))
tab_lower = F.lower(F.col("objeto"))

# classe pela coluna (mais especifica, tem prioridade)
por_coluna = F.lit(None).cast("string")
for nome, padrao in reversed(COLUNA_CLASSE):
    por_coluna = F.when(col_lower.rlike(padrao), F.lit(nome)).otherwise(por_coluna)

# classe herdada da tabela
por_tabela = F.lit(None).cast("string")
for nome, padrao in reversed(TABELA_CLASSE):
    por_tabela = F.when(tab_lower.rlike(padrao), F.lit(nome)).otherwise(por_tabela)

col = (bronze("colunas")
       .filter(~F.col("schema_nome").isin(SISTEMA))
       .withColumn("classe_por_coluna", por_coluna)
       .withColumn("classe_por_tabela", por_tabela)
       .withColumn("coluna_tecnica", col_lower.rlike(TECNICAS))
       .withColumn("classe_dado",
                   F.coalesce(F.col("classe_por_coluna"),
                              F.when(~F.col("coluna_tecnica"), F.col("classe_por_tabela"))))
       .withColumn("origem_classificacao",
                   F.when(F.col("classe_por_coluna").isNotNull(), "coluna")
                    .when(F.col("classe_dado").isNotNull(), "tabela"))
       .withColumn("is_dado_pessoal", F.col("classe_dado").isNotNull())
       .withColumn("is_dado_saude", F.col("classe_dado").isin(SAUDE))
       .withColumn("is_dado_sensivel", F.col("classe_dado").isin(SENSIVEIS))
       .withColumn("nullable", booleano("is_nullable"))
       .withColumn("coluna_anon",
                   F.concat(F.lit("c_"),
                            F.substring(F.sha2(F.concat_ws("|", "schema_nome", "objeto", "coluna"), 256), 1, 8)))
       .dropDuplicates(["schema_nome", "objeto", "coluna"]))

silver_col = salvar(col.select("schema_nome", "objeto", "coluna", "coluna_anon", "tipo_dado",
                              "nullable", "classe_dado", "origem_classificacao",
                              "is_dado_pessoal", "is_dado_saude", "is_dado_sensivel", "_snapshot"),
                    "coluna",
                    "Um registro por coluna, com classificacao LGPD. classe_dado vem do nome da "
                    "coluna quando ha padrao especifico, senao e herdada do nome da tabela - "
                    "colunas tecnicas nao herdam. is_dado_saude marca o art. 11; is_dado_sensivel "
                    "abrange o art. 5, II. Origem: bronze.colunas.")

total = silver_col.count()
pessoais = silver_col.filter("is_dado_pessoal").count()
saude = silver_col.filter("is_dado_saude").count()
print(f"\n{pessoais}/{total} colunas classificadas como dado pessoal ({100*pessoais/total:.1f}%)")
print(f"{saude} colunas de dado de saude, em "
      f"{silver_col.filter('is_dado_saude').select('objeto').distinct().count()} tabelas")
display(silver_col.groupBy("classe_dado", "origem_classificacao").count().orderBy(F.desc("count")))

display(silver_col.groupBy("classe_dado", "is_dado_sensivel").count().orderBy(F.desc("count")))

workspace.silver.coluna                          1238

301/1238 colunas classificadas como dado pessoal (24.3%)
210 colunas de dado de saude, em 39 tabelas


classe_dado,origem_classificacao,count
null,null,937
saude_metrica,tabela,115
saude_clinica,tabela,58
credencial,coluna,31
saude_metrica,coluna,24
credencial,tabela,18
nome_pessoa,coluna,15
email,coluna,14
saude_clinica,coluna,13
telefone,coluna,5


classe_dado,is_dado_sensivel,count
null,null,937
saude_metrica,true,139
saude_clinica,true,71
credencial,false,49
nome_pessoa,false,15
email,false,14
telefone,false,5
endereco,false,3
nascimento,false,2
prof_saude,false,2


### Acuracia do classificador
Regex sobre nome de coluna erra nos dois sentidos. Amostro 40 colunas para estimar
precisao e recall em vez de apresentar a heuristica como se fosse exata.

In [0]:
# amostra deterministica: hash estavel da chave em vez de rand(), para que a mesma
# amostra saia em qualquer execucao e o numero do relatorio seja reproduzivel
amostra = (silver_col
           .filter(F.col("schema_nome").isin(EXPOSTOS))
           .withColumn("_ordem", F.sha2(F.concat_ws("|", "objeto", "coluna", F.lit("42")), 256))
           .orderBy("_ordem")
           .select("objeto", "coluna", "tipo_dado", "classe_dado",
                   "origem_classificacao", "is_dado_saude")
           .limit(40))
display(amostra)

# As 40 linhas acima foram rotuladas a mao sob a definicao ampla do art. 5, I da LGPD.
# Resultado: VP=12, FP=3, FN=12, VN=13 -> precisao 80,0% e recall 50,0%.
# O classificador e confiavel quando acusa, mas encontra metade do que existe.
# Os dois erros mais ilustrativos estao discutidos no README:
#   FN: profiles.cycle_length_days (ciclo menstrual, nome neutro)
#   FP: checkins.training_type (o usuario torna publico por escolha propria)

objeto,coluna,tipo_dado,classe_dado,origem_classificacao,is_dado_saude
goals,user_id,uuid,null,null,null
checkin_groups,created_at,timestamp with time zone,null,null,null
assigned_workouts,notes,text,saude_metrica,tabela,true
photo_trackings,user_id,uuid,null,null,null
app_events,event_type,text,null,null,null
invite_links,link_type,text,credencial,tabela,false
nutrition_plans,name,text,nome_pessoa,coluna,false
ai_chat_threads,created_at,timestamp with time zone,null,null,null
body_measurements,notes,text,saude_metrica,tabela,true
access_audit_events,device_id,text,null,null,null


## Politicas de RLS

`pg_policies.roles` vem como array. No CSV virou string separada por `;` porque
usei `array_to_string` na extracao - sem isso o parser de CSV quebrava nas virgulas
internas. Aqui desfaco com split + explode.

Nao levo a expressao SQL literal para a Silver. Ela revela a estrutura de
multi-tenancy do Pitaia, e o repositorio e publico. Derivo flags e descarto o texto.

In [0]:
pol = (bronze("politicas")
       .filter(~F.col("schema_nome").isin(SISTEMA))
       # explode precisa vir sozinho: Spark nao aceita gerador aninhado em outra
       # expressao (UNSUPPORTED_GENERATOR.NESTED_IN_EXPRESSIONS). Trim vem na linha seguinte.
       .withColumn("role_nome", F.explode(F.split(F.coalesce(F.col("roles"), F.lit("")), ";")))
       .withColumn("role_nome", F.trim(F.col("role_nome")))
       .filter(F.col("role_nome") != "")
       .withColumn("permissiva", F.upper(F.trim(F.col("permissive"))) == "PERMISSIVE")
       # politica so com WITH CHECK tem qual nulo - uso coalesce em vez de descartar
       .withColumn("expr", F.lower(F.coalesce(F.col("expressao_using"),
                                              F.col("expressao_check"), F.lit(""))))
       .withColumn("so_check", F.col("expressao_using").isNull()
                               & F.col("expressao_check").isNotNull())
       .withColumn("libera_tudo", F.regexp_replace(F.col("expr"), r"[\s()]", "") == "true")
       .withColumn("filtra_uid", F.col("expr").contains("auth.uid()"))
       .withColumn("filtra_jwt", F.col("expr").contains("auth.jwt()")
                                 | F.col("expr").contains("auth.role()"))
       .withColumn("filtra_tenant",
                   F.col("expr").rlike(r"tenant|paciente_id|patient_id|user_id|owner_id|"
                                       r"medico_id|profissional_id|clinica_id|org_id"))
       .withColumn("tem_isolamento", F.col("filtra_uid") | F.col("filtra_jwt") | F.col("filtra_tenant"))
       .dropDuplicates(["schema_nome", "objeto", "politica", "role_nome"]))

salvar(pol.select("schema_nome", "objeto", "politica", "role_nome", "comando", "permissiva",
                  "so_check", "libera_tudo", "filtra_uid", "filtra_jwt", "filtra_tenant",
                  "tem_isolamento", "_snapshot"),
       "politica",
       "Um registro por politica x role (array explodido). tem_isolamento indica se a "
       "expressao amarra a linha ao usuario, ao JWT ou a uma coluna de tenant. libera_tudo "
       "marca USING(true). A expressao literal nao e publicada. Origem: bronze.politicas.")

display(spark.table(f"{SILVER}.politica")
             .groupBy("role_nome", "tem_isolamento").count().orderBy("role_nome"))

workspace.silver.politica                         209


role_nome,tem_isolamento,count
authenticated,true,95
authenticated,false,5
public,true,102
public,false,7


## Grants, funcoes, buckets, extensoes, constraints

In [0]:
salvar(bronze("grants")
       .filter(~F.col("schema_nome").isin(SISTEMA))
       .select("role_nome", "schema_nome", "objeto", "privilegio", "_snapshot")
       .dropDuplicates(["role_nome", "schema_nome", "objeto", "privilegio"]),
       "grant_role",
       "Um registro por role x objeto x privilegio, deduplicado - heranca de role repete "
       "a mesma tupla. Origem: bronze.grants.")

salvar(bronze("funcoes")
       .withColumn("security_definer", booleano("security_definer"))
       .withColumn("tem_search_path", F.coalesce(F.col("config"), F.lit("")).contains("search_path"))
       .select("schema_nome", "funcao", "security_definer", "tem_search_path", "linguagem", "_snapshot")
       # NAO deduplicar por (schema_nome, funcao). O perfil de duplicatas acusou 30
       # repeticoes, que sao overloads do Postgres: mesma funcao, assinaturas diferentes,
       # cada uma com suas proprias flags. Descartar 30 esconderia um SECURITY DEFINER
       # vulneravel e daria falso negativo na R04. Dedup so de linha integralmente igual.
       .dropDuplicates(),
       "funcao",
       "Uma linha por funcao. Funcoes sobrecarregadas (mesmo nome, assinaturas diferentes) "
       "ficam em linhas distintas de proposito: cada assinatura tem suas proprias flags de "
       "seguranca. security_definer executa com privilegio do criador e contorna RLS; sem "
       "search_path fixado isso viabiliza escalada de privilegio. O corpo da funcao nao foi "
       "coletado. Origem: bronze.funcoes.")

salvar(bronze("buckets").withColumn("publico", booleano("publico"))
       .select("bucket_id", "bucket_nome", "publico", "_snapshot"),
       "bucket",
       "Um registro por bucket do Storage. publico=true libera leitura anonima de todo o "
       "conteudo - critico no Pitaia, onde bucket costuma guardar exame e laudo. "
       "Origem: bronze.buckets.")

salvar(bronze("extensoes").select("extensao", "schema_nome", "versao", "_snapshot"),
       "extensao",
       "Um registro por extensao instalada. Superficie de cadeia de suprimentos, "
       "OWASP A03:2025. Origem: bronze.extensoes.")

salvar(bronze("constraints")
       .filter(~F.col("schema_nome").isin(SISTEMA))
       .select("schema_nome", "objeto", "constraint_nome", "tipo", "_snapshot")
       .dropDuplicates(["schema_nome", "objeto", "constraint_nome"]),
       "constraint_tabela",
       "Um registro por constraint de PK, UNIQUE ou FK. Usada para achar tabela sem chave "
       "primaria. Origem: bronze.constraints.")

workspace.silver.grant_role                      3826
workspace.silver.funcao                           167
workspace.silver.bucket                             7
workspace.silver.extensao                           8
workspace.silver.constraint_tabela                270


DataFrame[schema_nome: string, objeto: string, constraint_nome: string, tipo: string, _snapshot: string]